In [1]:
import os
import sys
# Get project root (one directory up from notebooks/)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))


In [2]:
from typing import List, Dict
from sqlalchemy.orm import Session
from app.ml.indoBERT.indobert_model import IndoBERTModel
from app.models import JobPosting, JobPostingQuestion, JobPostingEmbedding
import numpy as np
import json

class JobPostingService:
    """
    Service for creating JD profiles:
    1. Parse JD into structured competencies
    2. Generate and cache embeddings
    3. Map questions to competencies
    """

    def __init__(self, model: IndoBERTModel, db: Session):
        self.model = model
        self.db = db

    def create_job_posting_profile(
        self,
        job_posting: JobPosting,
        questions: List[JobPostingQuestion],
    )->JobPosting:
        """
        Create complete JD profile with embeddings

        Args:
            job_posting: JobPosting instance with description
            questions: List of HR questions with weights and competency mappings

        Returns:
            Updated JobPosting with embeddings cached
        """
        # Step 1: Normalize question weights
        # total_weight = sum(q.weight for q in questions)
        # for q in questions:
        #     q.weight = q.weight / total_weight
        #     self.db.add(q)
        #
        # Step 2: Generate and cache embeddings for job posting competency:
        #   Requirements
        #   Responsibility
        #   Qualifications
        #   Required Skills
        #   Preferred Skills
        self._generate_embeddings(job_posting)

        self.db.commit()
        self.db.refresh(job_posting)
        return job_posting

    def get_job_posting_embeddings_for_questions(
        self,
        job_posting_id: str
    ) -> Dict[str, List[np.ndarray]]:
        '''
        Get Job Posting Embeddings organized by question ID

        Args:
            job_posting_id: Job Posting ID

        Returns:
           Dict mapping question_id to list of relevant Job Posting embeddings
        '''
        job_posting = self.db.query(JobPosting).filter(
            JobPosting.id == job_posting_id
        ).first()

        if not job_posting:
            return {}

        # Get all embeddings
        jp_embeddings = self.db.query(JobPostingEmbedding).filter(
            JobPostingEmbedding.job_posting_id == job_posting_id
        ).all()

        # Get all questions
        questions = self.db.query(JobPostingQuestion).filter(
            JobPostingQuestion.job_posting_id == job_posting_id
        ).all()

        # Organize by question based on mapped_competencies
        question_embeddings = {}

        for question in questions:
            qid = question.id
            mapped_comps = question.mapped_competencies

            # Get embeddings for mapped competencies
            relevant_embeddings = []
            for emb in jp_embeddings:
                if emb.competency_id in mapped_comps:
                    relevant_embeddings.append(
                        np.array(emb.embedding, dtype=np.float32)
                    )
            question_embeddings[qid] = relevant_embeddings
        return question_embeddings

    def _generate_embeddings(self, job_posting: JobPosting):
        """Generate embeddings for all competencies"""

        embedding_data = []

        # Responsibilities
        job_posting.responsibilities = self._safe_json_parse(job_posting.responsibilities)
        embedding_data.extend(self._process_embedding_items(
            job_posting.id,
            job_posting.responsibilities,
            'responsibilities',
        ))

        # Requirements
        job_posting.requirements = self._safe_json_parse(job_posting.requirements)
        embedding_data.extend(self._process_embedding_items(
            job_posting.id,
            job_posting.requirements,
            'requirements',
        ))

        # Qualifications
        job_posting.qualifications = self._safe_json_parse(job_posting.qualifications)
        embedding_data.extend(self._process_embedding_items(
            job_posting.id,
            job_posting.qualifications,
            'qualifications',
        ))

        # Preferred skills
        job_posting.preferred_skills = self._safe_json_parse(job_posting.preferred_skills)
        embedding_data.extend(self._process_embedding_items(
            job_posting.id,
            job_posting.preferred_skills,
            'preferred_skills',
        ))

        # Required skills
        job_posting.required_skills = self._safe_json_parse(job_posting.required_skills)
        embedding_data.extend(self._process_embedding_items(
            job_posting.id,
            job_posting.required_skills,
            'required_skills',
        ))

        # Save to database
        for emb_dict in embedding_data:
            jd_emb = JobPostingEmbedding(
                job_posting_id=job_posting.id,
                competency_type=emb_dict["competency_type"],
                competency_id=emb_dict["competency_id"],
                text=emb_dict["text"],
                embedding=emb_dict["embedding"]
            )
            self.db.add(jd_emb)

    def _process_embedding_items(self,job_posting_id, data_list, data_type):
        """
        Generate embeddings for a list of text items.

        :param job_posting_id: Job Posting ID
        :param data_list: list of dicts or strings (e.g. responsibilities, requirements, etc.)
        :param data_type: string to label the type (e.g. 'responsibility', 'requirement')
        :return: list of dicts with id, text, and embedding
        """
        embedding_data = []

        for idx, item in enumerate(data_list or []):
            text_value = item.get("value") if isinstance(item, dict) else str(item)
            if not text_value:
                continue  # skip empty

            existing = self.db.query(JobPostingEmbedding).filter(
                JobPostingEmbedding.text == text_value,
                JobPostingEmbedding.competency_type == data_type,
                JobPostingEmbedding.job_posting_id == job_posting_id
            ).first()

            if existing:
                continue # skip
            embedding = self.model.encode(text_value)[0]

            embedding_data.append({
                "competency_type": data_type,
                "competency_id": item.get("id") if isinstance(item, dict) else f"{data_type}_{idx+1}",
                "text": text_value,
                "embedding": embedding.tolist()
            })

        return embedding_data

    def _safe_json_parse(self, value, default=None):
        """
        Safely parse a JSON string to Python object.
        - If value is already a dict/list, it’s returned as-is.
        - If parsing fails, returns `default`.
        """
        if default is None:
            default = []

        if isinstance(value, (dict, list)):
            return value

        if isinstance(value, str):
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                print(f"❌ Failed to parse JSON: {value}")
                return default

        return default


DATABASE_URL sqlite:////home/fariz/Documents/project/HRIS/ai-service/notebooks/data/dev.db


In [8]:
import traceback
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from app.services.calibration_service import CalibrationService
from app.core.ml_loader import get_model
import pandas as pd
import uuid

# Ensure we're in the project root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Create SQLite engine manually
DATABASE_URL = "sqlite:///./data/dev.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})

# Create your own session
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
db = SessionLocal()
job_posting_id = uuid.UUID("400a399d-ec56-4df0-a2de-33acd3349b67")


print(f"🚀 Processing JD profile for job {job_posting_id}...")
# Get job posting
job_posting = db.query(JobPosting).filter(
    JobPosting.id == job_posting_id
).first()


if not job_posting:
    raise ValueError(f"Job posting {job_posting_id} not found")

print("================= JOB POSTING ================= ")
data = job_posting.__dict__.copy()
data.pop('_sa_instance_state', None)  # remove internal SQLAlchemy state
pd.DataFrame([data])



🚀 Processing JD profile for job 400a399d-ec56-4df0-a2de-33acd3349b67...
================= JOB POSTING ================= 


,id,title,description,qualifications,preferred_skills,flag_threshold,created_at,updated_at,requirements,responsibilities,required_skills,shortlist_threshold,status
0,400a399d-ec56-4df0-a2de-33acd3349b67,Staf Administrasi Biro Pembelajaran,Kami mencari Staf Administrasi yang teliti dan...,"[{'id': 'qualification_1', 'value': 'Teliti, d...","[{'id': 'preferred_skill_1', 'value': 'Pengala...",None,2025-11-03 03:09:35.709983,2025-11-03 03:01:52.261738,"[{'id': 'requirement_1', 'value': 'Minimal lul...","[{'id': 'responsibility_1', 'value': 'Mengelol...","[{'id': 'required_skill_1', 'value': 'Microsof...",None,active


In [10]:
# Load model
model = get_model()

# Step 1: Create Job Posting profile (parse + embed)
print(f"  📝 Parsing JD and generating embeddings...")
questions = [
        {
            "id": "588e4c1c-9c09-4b28-b91e-819ffeedee3f",
			"job_posting_id": "400a399d-ec56-4df0-a2de-33acd3349b67",
            "question": "Ceritakan pengalaman kerja terdahulu. Anda boleh menceritakan relevansi pengalaman kerja dulu dengan lowongan kerja yang Bapak/Ibu lamar.",
            "description": "",
            "weight": 0.3,
            "mapped_competencies": [
                "requirement_4",
                "preferred_skill_3"
            ],
			"weight_version": 1
        },
        {
            "id": "0c562fb9-3f92-4c0b-8790-bc8011eea8b5",
			"job_posting_id": "400a399d-ec56-4df0-a2de-33acd3349b67",
            "question": "Apa motivasi Bapak/Ibu untuk bekerja di Universitas Trilogi?",
            "description": "",
            "weight": 0.2,
            "mapped_competencies": [
                "qualification_2",
                "qualification_3"
            ],
			"weight_version": 1
        },
        {
            "id": "46b63e3a-048f-44a2-981d-0d863935fb7d",
			"job_posting_id": "400a399d-ec56-4df0-a2de-33acd3349b67",
            "question": "Apa yang Bapak/Ibu ketahui tentang posisi ini?",
            "description": "",
            "weight": 0.2,
            "mapped_competencies": [
                "responsibility_1",
                "responsibility_4"
            ],
			"weight_version": 1
        },
        {
            "id": "4f08f721-4850-4976-b329-e8c09eae14d2",
			"job_posting_id": "400a399d-ec56-4df0-a2de-33acd3349b67",
            "question": "Apa rencana pengembangan ke depannya apabila Anda diterima dalam posisi ini?",
            "description": "",
            "weight": 0.15,
            "mapped_competencies": [
                "qualification_3",
                "preferred_skill_1"
            ],
			"weight_version": 1
        },
        {
            "id": "caa65eb0-4a2d-4ece-81e1-a6e47eb31a42",
			"job_posting_id": "400a399d-ec56-4df0-a2de-33acd3349b67",
            "question": "Jika Anda diterima, apa yang Anda butuhkan dari Biro Sumber Daya Manusia untuk mengembangkan diri Anda?",
            "description": "",
            "weight": 0.15,
            "mapped_competencies": [
                "qualification_1",
                "required_skill_1"
            ],
			"weight_version": 1
        }
    ]
jpp_service = JobPostingService(model, db)
question_list: List[JobPostingQuestion] = []
for q in questions:
    print(q)
    question_list.append(JobPostingQuestion(
        id=uuid.UUID(q['id']),
        job_posting_id=q['job_posting_id'],
        question=q['question'],
        weight=q['weight'],
        mapped_competencies=q['mapped_competencies'],
        weight_version=q['weight_version'],
    ))
print(question_list)
job_posting = jpp_service.create_job_posting_profile(job_posting, question_list)

embeddings_count = db.query(JobPostingEmbedding).filter(
    JobPostingEmbedding.job_posting_id == job_posting_id
).count()

  📝 Parsing JD and generating embeddings...
{'id': '588e4c1c-9c09-4b28-b91e-819ffeedee3f', 'job_posting_id': '400a399d-ec56-4df0-a2de-33acd3349b67', 'question': 'Ceritakan pengalaman kerja terdahulu. Anda boleh menceritakan relevansi pengalaman kerja dulu dengan lowongan kerja yang Bapak/Ibu lamar.', 'description': '', 'weight': 0.3, 'mapped_competencies': ['requirement_4', 'preferred_skill_3'], 'weight_version': 1}
{'id': '0c562fb9-3f92-4c0b-8790-bc8011eea8b5', 'job_posting_id': '400a399d-ec56-4df0-a2de-33acd3349b67', 'question': 'Apa motivasi Bapak/Ibu untuk bekerja di Universitas Trilogi?', 'description': '', 'weight': 0.2, 'mapped_competencies': ['qualification_2', 'qualification_3'], 'weight_version': 1}
{'id': '46b63e3a-048f-44a2-981d-0d863935fb7d', 'job_posting_id': '400a399d-ec56-4df0-a2de-33acd3349b67', 'question': 'Apa yang Bapak/Ibu ketahui tentang posisi ini?', 'description': '', 'weight': 0.2, 'mapped_competencies': ['responsibility_1', 'responsibility_4'], 'weight_version

PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (builtins.AttributeError) 'str' object has no attribute 'hex'
[SQL: INSERT INTO job_posting_question (id, job_posting_id, question, weight, mapped_competencies, weight_version, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: [{'job_posting_id': '400a399d-ec56-4df0-a2de-33acd3349b67', 'question': 'Ceritakan pengalaman kerja terdahulu. Anda boleh menceritakan relevansi pengal ... (56 characters truncated) ... ar.', 'id': '588e4c1c-9c09-4b28-b91e-819ffeedee3f', 'weight': 0.3, 'weight_version': 1, 'mapped_competencies': ['requirement_4', 'preferred_skill_3']}, {'job_posting_id': '400a399d-ec56-4df0-a2de-33acd3349b67', 'question': 'Apa motivasi Bapak/Ibu untuk bekerja di Universitas Trilogi?', 'id': '0c562fb9-3f92-4c0b-8790-bc8011eea8b5', 'weight': 0.2, 'weig

In [ ]:
try:

    # Load model
    model = get_model()

    # Step 1: Create Job Posting profile (parse + embed)
    print(f"  📝 Parsing JD and generating embeddings...")
    jpp_service = JobPostingService(model, db)
    question_list: List[JobPostingQuestion] = []
    for q in questions:
        print(q)
        question_list.append(JobPostingQuestion(
            id=q['id'],
            job_posting_id=q['job_posting_id'],
            question=q['question'],
            weight=q['weight'],
            mapped_competencies=q['mapped_competencies'],
            weight_version=q['weight_version'],
        ))
    print(question_list)
    job_posting = jpp_service.create_job_posting_profile(job_posting, question_list)

    embeddings_count = db.query(JobPostingEmbedding).filter(
        JobPostingEmbedding.job_posting_id == job_posting_id
    ).count()

    print(f"  ✅ Generated {embeddings_count} embeddings")

    # Step 2: Calibrate thresholds
    print(f"  🎯 Calibrating thresholds...")
    calibration_service = CalibrationService(db)

    try:
        shortlist_th, flag_th = calibration_service.calibrate_thresholds(
            job_posting_id,
            top_k=5
        )
        print(f"  ✅ Thresholds: shortlist={shortlist_th:.3f}, flag={flag_th:.3f}")
    except Exception as calib_error:
        print(f"  ⚠️  Calibration warning: {calib_error}")
        # Use defaults
        shortlist_th = 0.75
        flag_th = 0.25
        job_posting.shortlist_threshold = shortlist_th
        job_posting.flag_threshold = flag_th
        db.commit()
        print(f"  ℹ️  Using default thresholds")

    # Step 3: Mark as active
    job_posting.status = "active"
    db.commit()
    db.refresh(job_posting)

    print(f"✅ JD profile processing complete for job {job_posting_id}")

    return {
        "status": "success",
        "job_posting_id": job_posting_id,
        "embeddings_count": embeddings_count,
        "shortlist_threshold": float(shortlist_th),
        "flag_threshold": float(flag_th),
        "competencies": {
            "responsibilities": len(job_posting.responsibilities or []),
            "required_skills": len(job_posting.required_skills or []),
            "preferred_skills": len(job_posting.preferred_skills or []),
            "qualifications": len(job_posting.qualifications or [])
        }
    }

except Exception as e:
    print(f"❌ Error processing JD profile: {e}")
    print(traceback.format_exc())

    # Mark job as failed
    try:
        job_posting = db.query(JobPosting).filter(
            JobPosting.id == job_posting_id
        ).first()
        if job_posting:
            job_posting.status = "failed"
            db.commit()
    except: pass
finally:
    db.close()
